# Phase 6: Advanced Hyperparameter Tuning

we will use **Optuna** to tune **LightGBM**, **CatBoost**, and **XGBoost** 

In [1]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import mlflow
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold

sys.path.append('../')
from src.metrics import zindi_score

optuna.logging.set_verbosity(optuna.logging.INFO)

/Users/USER/Desktop/zindi/ml_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data & One-Hot Encode

In [2]:
train_df = pd.read_parquet('../data/features/train_features.parquet', engine='fastparquet')
raw_train = pd.read_csv('../data/Train.csv')
target = raw_train['liquidity_stress_next_30d']

X = train_df.drop(columns=['ID', 'liquidity_stress_next_30d'])

# One-Hot Encoding for Trees
cat_cols = X.select_dtypes(include=['object', 'category']).columns
if len(cat_cols) > 0:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.astype({col: int for col in X.select_dtypes(include=bool).columns})

print(f"Training Data Shape: {X.shape}")

Training Data Shape: (40000, 228)


## 2. LightGBM Optuna Objective

In [3]:
def objective_lgb(trial):
    param = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'random_state': 42,
        'n_estimators': 300,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    
    for train_idx, val_idx in skf.split(X, target):
        model = lgb.LGBMClassifier(**param)
        model.fit(X.iloc[train_idx], target.iloc[train_idx])
        oof_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        
    loss, _ = zindi_score(target, oof_preds)
    return loss

## 3. CatBoost Optuna Objective

In [4]:
def objective_cb(trial):
    param = {
        'loss_function': 'Logloss',
        'verbose': False,
        'random_seed': 42,
        'iterations': 300,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    
    for train_idx, val_idx in skf.split(X, target):
        model = cb.CatBoostClassifier(**param)
        model.fit(X.iloc[train_idx], target.iloc[train_idx])
        oof_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        
    loss, _ = zindi_score(target, oof_preds)
    return loss

## 4. XGBoost Optuna Objective

In [5]:
def objective_xgb(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'verbosity': 0,
        'random_state': 42,
        'n_estimators': 300,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    
    for train_idx, val_idx in skf.split(X, target):
        model = xgb.XGBClassifier(**param)
        model.fit(X.iloc[train_idx], target.iloc[train_idx])
        oof_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        
    loss, _ = zindi_score(target, oof_preds)
    return loss

## 5. Execute Optuna Studies
>  This cell will take time to run as it trains 60 full cross-validation models so just go watch ball, lol

In [6]:
print("------ TUNING LIGHTGBM ------")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=20)

print("\n------ TUNING CATBOOST ------")
study_cb = optuna.create_study(direction='minimize')
study_cb.optimize(objective_cb, n_trials=20)

print("\n------ TUNING XGBOOST ------")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=20)

print("\n\n" + "*"*50)
print("ALL TUNING COMPLETE! HERE ARE YOUR BEST PARAMETERS:")
print("*"*50)

print(f"\n1. LightGBM (Best LogLoss: {study_lgb.best_value:.4f})")
print("lgb_params =", study_lgb.best_params)

print(f"\n2. CatBoost (Best LogLoss: {study_cb.best_value:.4f})")
print("cb_params =", study_cb.best_params)

print(f"\n3. XGBoost (Best LogLoss: {study_xgb.best_value:.4f})")
print("xgb_params =", study_xgb.best_params)

[I 2026-09-01 04:35:10,919] A new study created in memory with name: no-name-174d4540-e56d-4335-8413-7bd954f8d927


------ TUNING LIGHTGBM ------


[I 2026-09-01 04:35:44,784] Trial 0 finished with value: 0.2960275854244281 and parameters: {'learning_rate': 0.013036058065628756, 'num_leaves': 75, 'max_depth': 8, 'min_child_samples': 31, 'subsample': 0.7651224524633855, 'colsample_bytree': 0.8205190453381673}. Best is trial 0 with value: 0.2960275854244281.


--- Model Evaluation ---
Log Loss: 0.2960
ROC-AUC: 0.8735
------------------------


[I 2026-09-01 04:36:14,176] Trial 1 finished with value: 0.2837928371199554 and parameters: {'learning_rate': 0.07818532506015811, 'num_leaves': 81, 'max_depth': 10, 'min_child_samples': 28, 'subsample': 0.8605941375204382, 'colsample_bytree': 0.6210661865117374}. Best is trial 1 with value: 0.2837928371199554.


...

--- Model Evaluation ---
Log Loss: 0.2746
ROC-AUC: 0.8848
------------------------


[I 2026-09-01 05:11:02,219] Trial 18 finished with value: 0.27303966549405456 and parameters: {'learning_rate': 0.03994135734577703, 'max_depth': 8, 'min_child_weight': 14, 'subsample': 0.8691524614712224, 'colsample_bytree': 0.7428785554534951}. Best is trial 18 with value: 0.27303966549405456.


--- Model Evaluation ---
Log Loss: 0.2730
ROC-AUC: 0.8864
------------------------


[I 2026-09-01 05:11:29,094] Trial 19 finished with value: 0.2747225881593436 and parameters: {'learning_rate': 0.036227156132481196, 'max_depth': 8, 'min_child_weight': 17, 'subsample': 0.8381767127074127, 'colsample_bytree': 0.7558205301866613}. Best is trial 18 with value: 0.27303966549405456.


--- Model Evaluation ---
Log Loss: 0.2747
ROC-AUC: 0.8851
------------------------


**************************************************
ALL TUNING COMPLETE! HERE ARE YOUR BEST PARAMETERS:
**************************************************

1. LightGBM (Best LogLoss: 0.2711)
lgb_params = {'learning_rate': 0.055448371170477934, 'num_leaves': 50, 'max_depth': 8, 'min_child_samples': 62, 'subsample': 0.6900164960230282, 'colsample_bytree': 0.6730449242220288}

2. CatBoost (Best LogLoss: 0.2740)
cb_params = {'learning_rate': 0.19845354600424955, 'depth': 5, 'l2_leaf_reg': 1.1335651261259694, 'subsample': 0.6049659306276488}

3. XGBoost (Best LogLoss: 0.2730)
xgb_params = {'learning_rate': 0.03994135734577703, 'max_depth': 8, 'min_child_weight': 14, 'subsample': 0.8691524614712224, 'colsample_bytree': 0.7428785554534951}


## Overfitting Check

In [7]:
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss

def check_overfitting():
    try:
        models_to_check = {}
        if 'model_lgb' in globals(): models_to_check['LightGBM'] = model_lgb
        if 'model_cb' in globals(): models_to_check['CatBoost'] = model_cb
        if 'model_xgb' in globals(): models_to_check['XGBoost'] = model_xgb
        if 'model' in globals() and 'LGBM' in str(type(model)): models_to_check['LightGBM'] = model
        
        if not models_to_check:
            print("No standard models found in memory to check.")
            return
            
        if 'X_train' not in globals() or 'X_val' not in globals():
            print("X_train or X_val not found in memory.")
            return
            
        train_losses = []
        val_losses = []
        names = []
        
        for name, m in models_to_check.items():
            train_p = m.predict_proba(X_train)[:, 1]
            val_p = m.predict_proba(X_val)[:, 1]
            train_losses.append(log_loss(y_train, train_p))
            val_losses.append(log_loss(y_val, val_p))
            names.append(name)
            
        x = range(len(names))
        width = 0.35
        
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar([i - width/2 for i in x], train_losses, width, label='Train Loss', color='#3498db')
        ax.bar([i + width/2 for i in x], val_losses, width, label='Val Loss', color='#e74c3c')
        
        ax.set_ylabel('Log Loss')
        ax.set_title('Overfitting Check: Train vs Validation Loss (Last Fold)')
        ax.set_xticks(x)
        ax.set_xticklabels(names)
        ax.legend()
        plt.show()
        
        for i, name in enumerate(names):
            print(f"{name} - Train: {train_losses[i]:.4f}, Val: {val_losses[i]:.4f}, Gap: {val_losses[i] - train_losses[i]:.4f}")
            
    except Exception as e:
        print(f"Could not run overfitting check: {e}")

check_overfitting()

No standard models found in memory to check.
